In [1]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import words

In [2]:
items_25 = pd.read_csv("../Final Project/items_2025.csv")
tesco_desc = pd.read_csv("../Final Project/tesco_parsed.csv")
asda_desc = pd.read_csv("../Final Project/asda_parsed_item_info.csv")
morrisons_desc = pd.read_csv("../Final Project/detailed_item_info_morrisons_2025-03-18_10-46-10.csv")
waitrose_desc = pd.read_csv("../Final Project/all_waitrose_item_info.csv")
sainsburys_desc = pd.read_csv("../Final Project/sainsburys_product_details.csv")

/var/folders/qz/pj0lh7817m3c9ydwgfrtbtf00000gn/T/ipykernel_42983/3287859164.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  tesco_desc = pd.read_csv("../Final Project/tesco_parsed.csv")


In [ ]:
cols = ['store_id','product_id','product_title','summary','description', 'category']
item_desc = pd.concat([tesco_desc[cols],sainsburys_desc[cols],asda_desc[cols],morrisons_desc[cols],waitrose_desc[cols]])

In [55]:
item_desc_merge = item_desc.merge(items_25[['store_id','product_id','product_breadcrumb','product_url']], how='left', on=['store_id', 'product_id'])

In [57]:
item_desc_merge['product_title'] = item_desc_merge['product_title'].fillna('')
item_desc_merge['summary'] = item_desc_merge['summary'].fillna('')
item_desc_merge['description'] = item_desc_merge['description'].fillna('')
item_desc_merge['product_breadcrumb'] = item_desc_merge['product_breadcrumb'].fillna('')

In [61]:
item_desc_merge = item_desc_merge[~item_desc_merge['product_url'].isna()]

In [64]:
item_desc_merge['TSDB'] = item_desc_merge['product_title'] + ' ' + item_desc_merge['summary'] + ' ' +  item_desc_merge['description'] + item_desc_merge['product_breadcrumb']

/var/folders/qz/pj0lh7817m3c9ydwgfrtbtf00000gn/T/ipykernel_42983/36383705.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_desc_merge['TSDB'] = item_desc_merge['product_title'] + ' ' + item_desc_merge['summary'] + ' ' +  item_desc_merge['description'] + item_desc_merge['product_breadcrumb']


In [70]:
# Ensure necessary nltk resources are available
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('words')

[nltk_data] Downloading package punkt to /Users/sambickel-
[nltk_data]     barlow/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/sambickel-
[nltk_data]     barlow/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/sambickel-
[nltk_data]     barlow/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /Users/sambickel-
[nltk_data]     barlow/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [71]:
stop_words = stopwords.words('english')
english_words = set(words.words())

In [72]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatize
    tokens = [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords
    tokens = [word for word in tokens if word in english_words]  # Remove non-english words
    return " ".join(tokens)

In [73]:
# Apply preprocessing
item_desc_merge['TSDB'] = item_desc_merge['TSDB'].astype(str)
item_desc_merge['processed'] = item_desc_merge['TSDB'].apply(preprocess_text)

/var/folders/qz/pj0lh7817m3c9ydwgfrtbtf00000gn/T/ipykernel_42983/3558394111.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_desc_merge['TSDB'] = item_desc_merge['TSDB'].astype(str)
/var/folders/qz/pj0lh7817m3c9ydwgfrtbtf00000gn/T/ipykernel_42983/3558394111.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  item_desc_merge['processed'] = item_desc_merge['TSDB'].apply(preprocess_text)


In [1]:
#item_desc_merge.to_csv('/Users/sambickel-barlow/Desktop/PP422/Final Project/all_data_processed.csv')